# A.18.1 The solve command

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The solve command has the form

```ampl
solve redirection opt ;
```

It causes AMPL to write the current translated problem to temporary files in directory $TMPDIR
(unless the current optimization problem has not changed since a previous write command), to
invoke a solver, and to attempt to read optimal primal and dual variables from the solver. If this
succeeds, the optimal variable values become available for printing and other uses. The optional
redirection is for the solver's standard output.

The current value of the solver option determines the solver that is invoked. Appending
'_oopt' to $solver gives the name of an option which, if defined with a nonempty string,
determines (by the first letter of the string) the style of temporary problem files written; otherwise,
AMPL uses its generic `binary` output format (style b). For example, if $solver is supersol,
then $supersol_oopt, if nonempty, determines the output style. The command-line option
'-o?' (A.23) shows a summary of the currently supported output styles.

AMPL passes two command-line arguments to the solver: the stub of the temporary files, and
the literal string -AMPL. AMPL expects the solver to write dual and primal variable values to file
stub . sol, preceded by commentary that, if appropriate, reports the objective value to
$objective_precision significant digits. In reading the solution, AMPL rounds the primal
variables to $solution_round places past the decimal point if $solution_round is an
integer, or to $solution_precision significant figures if $solution_precision is a
positive integer; the defaults for these options imply that no rounding is performed.

A variable always has a current value. A variable declaration or data section can specify the
initial value, which is otherwise 0. The option reset_initial_guesses controls the initial
guess conveyed to the solver. If option `reset` _ initial _ guesses has its default value of 0,
then the current variable values are conveyed as the initial guess. Setting option
`reset` _ initial _ guesses to 1 causes the original initial values to be sent. Thus
$reset_initial_guesses affects the starting guess for a second solve command, as well
as for an initial solve command that follows a solution command (described below).

A constraint always has an associated current dual variable value (Lagrange multiplier). The
initial dual value is 0 unless otherwise given in a data section or specified in the constraint's declaration
by a := initial_dual or a default initial_dual phrase. Whether a dual initial guess is
conveyed to solvers is governed by the option dual _ initial _ guesses. Its default value of 1
causes AMPL to pass the current dual variables (if $reset_initial_guesses is 0) or the
original initial dual variables to the solver; if $dual_initial_guesses is set to 0, AMPL will
omit initial values for the dual variables.

AMPL's presolve phase computes two sets of bounds on variables. The first set reflects any
sharpening of the declared bounds implied by eliminated constraints. The other set incorporates
sharpenings of the first set that presolve deduces from constraints it cannot eliminate from the
problem. The problem has the same solutions with either set of bounds, but solvers that use
active-set strategies (such as the simplex method) may have more trouble with degeneracies with
the sharpened bounds. Solvers often run faster with the first set, but sometimes run faster with the
second. By default, AMPL passes the first set of bounds, but if option var_bounds is 2, AMPL
passes the second set. The .lb and .ub suffixes for variables always reflect the current setting of
$var_bounds; .lb1 and .ub1 are for the first set, .lb2 and .ub2 for the second set.

If the output style is m, AMPL writes a conventional MPS file, and the value of option
integer_markers determines whether integer variables are indicated in the MPS file by
'INTORG' and 'INTEND' 'MARKER' lines. By default, $integer_markers is 1, causing
these lines to be written; specifying

```ampl
option integer_markers 0;
```

causes AMPL to omit the 'MARKER' lines.

The option `relax_integrality` causes integer and `binary` attributes of variables to
be ignored in solve and write commands. It is also possible to control this by setting the
.relax suffix of a variable (A.11).

By default, values of suffixes of type IN or INOUT (A.11.1) are sent to the solver, and updated
values for suffixes of type OUT or INOUT are obtained from the solver, but the sending and receiving
of suffix values can be controlled by setting option send_suffixes suitably: if
$send_suffixes is 1 or 3, suffix values are sent to the solver; and if $send_suffixes is 2
or 3, then updated suffix values are requested from the solver.

Whether .sstatus values (A.11.2) are sent to the solver is determined by options
send_suffixes and send_statuses; setting $send_statuses to 0 causes .sstatus
values not to be sent when $send_suffixes permits sending other suffixes.

Most solvers supply a value for AMPL's built-in symbolic parameter solve_message.
AMPL prints the updated solve_message by default, but setting option solver _ msg to 0
suppresses this printing. Most solvers also supply a numeric return code solve_result_num,
which has a corresponding symbolic value solve_result that is derived from
solve_result_num and $solve_result_table analogously to symbolic suffix values
(A.11.1).

By default AMPL permutes variables and constraints so solvers see the nonlinear ones first.
Some solvers require this, but with other solvers, occasionally it is useful to suppress these permutations
by setting option nl_permute suitably. It is the sum of

```ampl
1      to permute constraints
2      to permute variables
4      to permute objectives
```

and its default value is 3.

When complementarity constraints are present, the system of constraints is considered square
if the number of "inequality complements inequality" constraints plus the number of equations
equals the number of variables. Some complementarity solvers require square systems, so by
default AMPL warns about nonsquare systems. This can be changed by adjusting option
compl_warn, which is the sum of

```ampl
1      warn about nonsquare complementarity systems
2      warn and regard nonsquare complementarity systems as infeasible
4      disregard explicit matchings of variables to equations
```